# Data engenearing workbook

## Context:
My holdstaker needs to menage data from he's clients. 
I recived a pendrive with two main pathes, containg data from clients
Total archive first count:

In path 'NF' all files has the same prefix 'NF'
In path 'PlanilhaCorretores' file prefix are ['Prestação de Contas', 'Planilha Comissão']

## Taks:
### - Merge 'pdf' files from same client
      - Identify and Extract client name in all prefix
      - Join match files as Client Name.pdf

### - Create a table containing information about clients: name, processNumber, situation, responsableOne, nf(bool)
      - match = extract and compare name from prefix
      - name = originalName, processNumber = (get originalName and processNumber from table in pdf file)
      - situation,responsableOne = 'Planilha Comissão 'Name' (situation, responsableOne)
      - nf (for client in clients, if nf 'name' in nf[], return true, else, false)


### - Create a table containing information about cliente in NF: name, nf_number, date  

## 1 - Data Collect

In [ ]:
#1. Imports, Paths and Data Structures

import os
import re
import pandas as pd
import PyPDF2
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import sys

print("Imports completed successfully.")



In [ ]:
#2. Define Paths

import json
from pathlib import Path

# Pastas fora do repositório (dados reais não são versionados).
# Defina as variáveis de ambiente ou ajuste os valores padrão.
PASTA_DADOS = Path(os.getenv("PASTA_DADOS", "../dados"))   # PDFs de entrada
PASTA_SAIDA = Path(os.getenv("PASTA_SAIDA", "../saida"))   # relatórios e PDFs mesclados
# Correções manuais: JSON local, ignorado pelo Git (modelo: correcoes_notebook.example.json)
ARQUIVO_CORRECOES = Path(os.getenv("ARQUIVO_CORRECOES", "_local/correcoes_notebook.json"))

PATH = str(PASTA_DADOS)

print("Paths defined successfully.")


In [ ]:
#3. Data Structures

#data from NF
nf_table = {
    "origem": [],
    "nome_arquivo": [],
    "nome": [],
    "data": [],
}
#data from Prestação de Contas
pr_table = {
    "origem": [],
    "nome_arquivo": [],
    "nome": [],
    "nome_encontrado": [],
    "n_processo":[],
}
#data from Planilhas
pl_table = {
    "origem": [],
    "nome_arquivo": [],
    "nome": [],
    "corretor": [],
    "situação": []
}
#final data set - estrutura corrigida
final_table = {
    "nome": [],
    "origem_nf": [],      # Lista de caminhos de NF para este cliente
    "origem_pl": [],      # Lista de caminhos de Planilhas para este cliente
    "origem_pr": [],      # Lista de caminhos de Prestações para este cliente
    "data_nf": [],        # Data da NF
    "n_processo": [],     # Número do processo (da prestação)
    "situação": [],       # Situação (da planilha)
    "corretor": [],       # Corretor (da planilha)
    "nome_encontrado": [] # Nome encontrado no PDF da prestação
}

print("Data structures initialized successfully.")

In [ ]:
#2. Populate nf_table, pr_table, pl_table with file_name and full paths

# Limpar as listas antes de popular (evita duplicação ao executar múltiplas vezes)
nf_table["origem"].clear()
nf_table["nome_arquivo"].clear()
pl_table["origem"].clear()
pl_table["nome_arquivo"].clear()
pr_table["origem"].clear()
pr_table["nome_arquivo"].clear()

for fullPath in [PATH]:
    for file in os.listdir(fullPath):
        if file.lower().startswith('nf '):
            nf_table["origem"].append(fullPath + '\\' + file)
            nf_table["nome_arquivo"].append(file)
        if file.lower().startswith('planilha'):
            pl_table["origem"].append(fullPath + '\\' + file)
            pl_table["nome_arquivo"].append(file)
        if file.lower().startswith('prestação'):
            pr_table["origem"].append(fullPath + '\\' + file)
            pr_table["nome_arquivo"].append(file)

print('Resumo da Análise de Arquivos PDF:')

print(f'Total de NF: {len(nf_table["origem"])}')
print(f'Total de Planilhas: {len(pl_table["origem"])}')
print(f'Total de Prestações: {len(pr_table["origem"])}')

In [ ]:
#3. Function to process names in all file_names as 'nome' , wich will be used as key for matching

def processar_nome(nome):
    nome = nome.title()

    # Remove prefix 
    for prefix in ['Planilha Comissão ', 'Prestação De Contas ', 'Nf ']:
        if nome.startswith(prefix.title()):
            nome = nome.replace(prefix.title(), '')

    # Remove sufix
    nome = nome.replace('.Pdf', '')
    nome = re.sub(r'\s*\(.*?\)', '', nome).strip()

    # Treating 'Espólio' cases
    if nome.lower().startswith(("espólio", "espolio")):
        nome = re.sub(r'^(Esp[óo]lio( De)?)\s*', '', nome, flags=re.IGNORECASE)
        nome = nome.strip() + " - Espólio"

    return nome

### 1.2 NF

#### Colect nf emission date in open file

In [ ]:
#1.2 Try colect data from scanned PDF using OCR (SUCCESSFUL ATTEMPT)

# Tesseract executable path
TESSERACT_PATH = os.getenv("TESSERACT_CMD", r"C:\Program Files\Tesseract-OCR\tesseract.exe")
pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH


def extrair_texto_pdf_ocr(file):
    texto_completo = ""
    try:
        # Abre o PDF
        doc = fitz.open(file)
        

        for i, pagina in enumerate(doc):
            # 1. Zoom para melhorar a qualidade da imagem para o OCR
            zoom = 2 
            matriz = fitz.Matrix(zoom, zoom)
            pix = pagina.get_pixmap(matrix=matriz)
            
            # 2. Converte para imagem PIL
            imagem = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            
            # 3. Tenta fazer o OCR
            try:
                # Tenta ler em PORTUGUÊS primeiro
                texto_pagina = pytesseract.image_to_string(imagem, lang='eng')
            except pytesseract.TesseractError as e:
                # Se der erro de "language not found", cai aqui
                print(f"\nAviso: {e}. Tentando OCR em INGLÊS.")

            texto_completo += texto_pagina
            

        doc.close()

    except Exception as e:
        print(f"\nERRO CRÍTICO: {e}")
        print("\nVerifique se o caminho na linha 9 está apontando para onde você instalou o Tesseract.")

    return texto_completo


def normalizar_data_malformada(texto_data):
    """
    Tenta corrigir datas malformadas pelo OCR.
    Exemplos: '14112/2023', '2810512024', '28/105/2024'
    """
    # Remove espaços e caracteres extras
    texto_data = texto_data.strip()
    
    # Extrai apenas dígitos
    apenas_digitos = re.sub(r'\D', '', texto_data)
    
    # Caso 1: Exatamente 8 dígitos (ddmmyyyy sem separadores)
    if len(apenas_digitos) == 8:
        dia = apenas_digitos[0:2]
        mes = apenas_digitos[2:4]
        ano = apenas_digitos[4:8]
        return f"{dia}/{mes}/{ano}"
    
    # Caso 2: Mais de 8 dígitos (pode ter barras no lugar errado)
    # Tenta pegar os últimos 4 como ano, 2 anteriores como mês, 2 anteriores como dia
    if len(apenas_digitos) >= 8:
        # Pega os últimos 4 dígitos como ano
        ano = apenas_digitos[-4:]
        resto = apenas_digitos[:-4]
        
        # Se sobrou pelo menos 4 dígitos, tenta extrair dia e mês
        if len(resto) >= 4:
            mes = resto[-2:]
            dia = resto[-4:-2]
            return f"{dia}/{mes}/{ano}"
    
    return None


def extrair_data_texto(texto):
    # Múltiplos padrões de busca (do mais específico ao mais genérico)
    padroes = [
        # Padrão 1: Data e Hora de Emissão (formato correto)
        r"Data e Hora de Emiss[ãa]o.*?:?\s*\"?(\d{2}/\d{2}/\d{4})",
        
        # Padrão 2: Data com erros de OCR - aceita dígitos extras ou barras faltando
        r"Data e Hora de Emiss.{1,3}o.*?:?\s*[\n\\]*\s*\"?(\d{2,4}[/]?\d{2,4}[/]?\d{4})",
        
        # Padrão 3: Sequência de 8 dígitos após "Data" ou "Emiss"
        r"(?:Data|Emiss).{0,30}?[\n\\:]*\s*\"?(\d{8,10})",
        
        # Padrão 4: Data formato padrão em qualquer lugar
        r"(\d{2}/\d{2}/\d{4})",
        
        # Padrão 5: Sequência de 8 dígitos (último recurso)
        r"(\d{2}\d{2}\d{4})"
    ]
    
    for i, padrao in enumerate(padroes, 1):
        match = re.search(padrao, texto, re.IGNORECASE | re.DOTALL)
        if match:
            data_bruta = match.group(1)
            
            # Tenta normalizar se não estiver no formato correto
            if '/' not in data_bruta or len(data_bruta) != 10:
                data_normalizada = normalizar_data_malformada(data_bruta)
                if data_normalizada:
                    data_bruta = data_normalizada
            
            # Validação: verifica se está no formato correto e ano válido
            if re.match(r'\d{2}/\d{2}/\d{4}', data_bruta):
                try:
                    dia, mes, ano = map(int, data_bruta.split('/'))
                    # Valida ano, mês e dia
                    if 2020 <= ano <= 2030 and 1 <= mes <= 12 and 1 <= dia <= 31:
                        return data_bruta
                except:
                    continue
    
    return None  # Retorna None se não encontrar data válida

In [ ]:
# Clear nf_table data before populating
nf_table['nome'].clear()
nf_table['data'].clear()

In [ ]:
#2. Populate nf_table with key names anda extracted dates
nf_table['nome'] = [processar_nome(n) for n in nf_table['nome_arquivo']]

for o in nf_table['origem']:
    texto = extrair_texto_pdf_ocr(o)
    data = extrair_data_texto(texto)    
    nf_table["data"].append(data)

In [ ]:
#3. Display nf_table as DataFrame
nf_df = pd.DataFrame(nf_table)
nf_df.tail(30)

In [ ]:
#4. Display nf_table info
nf_df.info()

In [ ]:
# Filter nf_df to get names where data is null
nomes_sem_data = nf_df[nf_df['data'].isnull()]['nome_arquivo'].tolist()

print(f"Total de clientes sem data: {len(nomes_sem_data)}\n")
print("Lista de nomes com data == null:")
for nome in nomes_sem_data:
    print(f"  - {nome}")

In [ ]:
# Correção manual de datas que não foram extraídas pelo OCR
def carregar_correcoes(chave):
    if not ARQUIVO_CORRECOES.exists():
        print(f"Arquivo de correções não encontrado ({ARQUIVO_CORRECOES}); seguindo sem correções.")
        return {}
    return json.loads(ARQUIVO_CORRECOES.read_text(encoding="utf-8")).get(chave, {})

datas_manuais = carregar_correcoes("datas_manuais")

# Atualizar nf_table com as datas manuais
correcoes_aplicadas = 0
for i, nome in enumerate(nf_table['nome_arquivo']):
    if nf_table['data'][i] is None and nome in datas_manuais:
        nf_table['data'][i] = datas_manuais[nome]
        correcoes_aplicadas += 1
        print(f"✓ Data corrigida para '{nome}': {datas_manuais[nome]}")

print(f"\n✓ Total de correções aplicadas: {correcoes_aplicadas}")

# Atualizar DataFrame
nf_df = pd.DataFrame(nf_table)
print(f"✓ DataFrame atualizado. Clientes sem data agora: {nf_df['data'].isnull().sum()}")

In [ ]:
#5. Display nf_table info after manual corrections
nf_df.info()

### 1.3 - Planilhas
#### Colect 'Corretor' and 'Situação' in file_name

In [ ]:
#1. Colect data from file_name as 'texto'

def extrair_situacao_corretor(texto):
    """
    Extrai situação e corretor do formato: (Situação Nome_Corretor)
    Exemplos: (VT João Silva), (Vt João Alfredo), (2 Inct. Inês), (Inctr. Teles), (correção Rafael)
    """
    
    # Converte para upper case para normalizar
    texto_upper = texto.upper()

    # Múltiplos padrões de extração
    padroes = [
        # Padrão 1: 2 ou mais espaços separam situação e corretor
        r'\(\s*(.+?)\s{2,}(.+?)\s*\)',
        
        # Padrão 2: Número + situação + ponto + espaço + nome (ex: "2 Inct. Inês")
        r'\(\s*(\d+\s+[A-Z]+\.?)\s+([A-Z][^)]+?)\s*\)',
        
        # Padrão 3: Situação com ponto + espaço + nome (ex: "Inctr. Teles")
        r'\(\s*([A-Z]+\.?)\s+([A-Z][^)]+?)\s*\)',
        
        # Padrão 4: Palavra + espaço + nome (ex: "correção Rafael")
        r'\(\s*([A-Z][A-ZÀ-Ú]+)\s+([A-Z][^)]+?)\s*\)'
    ]
    
    for padrao in padroes:
        match = re.search(padrao, texto_upper)
        if match:
            situacao = match.group(1).strip()
            corretor = match.group(2).strip()
            
            # Validação: situação não deve ser muito longa (evita falsos positivos)
            if len(situacao) <= 30 and len(corretor) >= 2:
                return situacao, corretor
    
    print(f"Verificar formato em: {texto[:50]}...")
    return None, None

def extrair_situacao(texto):
    situacao, _ = extrair_situacao_corretor(texto)
    return situacao

def extrair_corretor(texto):
    _, corretor = extrair_situacao_corretor(texto)
    return corretor

In [ ]:
pl_table['nome'].clear()
pl_table['corretor'].clear()
pl_table['situação'].clear()

In [ ]:
#2. Populate pl_table with key names, corretor and situação'
pl_table['nome'] = [processar_nome(n) for n in pl_table['nome_arquivo']]
pl_table['corretor'] = [extrair_corretor(t) for t in pl_table['origem']]
pl_table['situação'] = [extrair_situacao(t) for t in pl_table['origem']]

In [ ]:
#3. Display pl_table as DataFrame
pl_df = pd.DataFrame(pl_table)
pl_df.head()

In [ ]:
#4. Display pl_table info
pl_df.info()

In [ ]:
corretor_nao_encontrado = pl_df[pl_df['corretor'].isnull()]['nome_arquivo'].tolist()

for a in corretor_nao_encontrado:
    print(f"Corretor não encontrado em: {a}")

In [ ]:
def carregar_correcoes(chave):
    if not ARQUIVO_CORRECOES.exists():
        print(f"Arquivo de correções não encontrado ({ARQUIVO_CORRECOES}); seguindo sem correções.")
        return {}
    return json.loads(ARQUIVO_CORRECOES.read_text(encoding="utf-8")).get(chave, {})

corretor_nao_encontrado = carregar_correcoes("corretor_nao_encontrado")

correcoes_aplicadas_corretor = 0
for i, nome in enumerate(pl_table['nome_arquivo']):
    if pl_table['corretor'][i] is None and nome in corretor_nao_encontrado:
        pl_table['corretor'][i] = corretor_nao_encontrado[nome]
        correcoes_aplicadas_corretor += 1
        print(f"✓ Corretor corrigido para '{nome}': {corretor_nao_encontrado[nome]}")

print(f"\n✓ Total de correções aplicadas: {correcoes_aplicadas_corretor}")

# Atualizar DataFrame
pl_df = pd.DataFrame(pl_table)
print(f"✓ DataFrame atualizado. Clientes sem corretor agora: {pl_df['corretor'].isnull().sum()}")

In [ ]:
pl_df['situação'].value_counts()

### 1.4 - Prestação De Contas
#### Colect 'n_processo', 'nome_encontrado' in open file

In [ ]:
#1. Extract data from Prestação de Contas pdf files
from PyPDF2 import PdfReader

def extrair_dados_prestacao(caminho_pdf):
    """Extrai nome do cliente e número do processo de um arquivo PDF"""
    try:
        reader = PdfReader(caminho_pdf)
        texto = ''.join([page.extract_text() or '' for page in reader.pages])

        # Extrair nome do cliente
        nome_match = re.search(r'À\s+([^\n]+)', texto)
        nome_cliente = nome_match.group(1).strip() if nome_match else "Nome não encontrado"

        # Extrair número do processo (padrões variados)
        processo_match = re.search(r'(?:Processo|N° Processo)[:\s]*([\d\.\-/]+)', texto, re.IGNORECASE)
        numero_processo = processo_match.group(1).strip() if processo_match else "Número não encontrado"

        return nome_cliente, numero_processo

    except Exception as e:
        print(f"Erro ao processar {os.path.basename(caminho_pdf)}: {str(e)}")
        return "Erro no processamento", ""





In [ ]:
pr_table['nome'].clear()
pr_table['nome_encontrado'].clear()
pr_table['n_processo'].clear()

In [ ]:
#2. Populate pr_table with key names, nome_encontrado and n_processo
pr_table['nome'] = [processar_nome(n) for n in pr_table['nome_arquivo']]
pr_table['nome_encontrado'] = [extrair_dados_prestacao(o)[0] for o in pr_table['origem']]
pr_table['n_processo'] = [extrair_dados_prestacao(o)[1] for o in pr_table['origem']]

In [ ]:
#3. Display pr_table as DataFrame
pr_df = pd.DataFrame(pr_table)
pr_df.head()

In [ ]:
#4. Display pr_table info
pr_df.info()

## Pipeline Final Table (Full Join)
Join data from 3 tables in final relatory

In [ ]:
# 1 Set names
todos_nomes = set()
todos_nomes.update(nf_table['nome'])
todos_nomes.update(pl_table['nome'])
todos_nomes.update(pr_table['nome'])

print(f"Total de clientes únicos encontrados: {len(todos_nomes)}")

In [ ]:
# 2. Group multiple origins by name
from collections import defaultdict

nf_por_nome = defaultdict(list)
pl_por_nome = defaultdict(list)
pr_por_nome = defaultdict(list)

# Agrupar NF por nome
for i, nome in enumerate(nf_table['nome']):
    nf_por_nome[nome].append({
        'origem': nf_table['origem'][i],
        'data': nf_table['data'][i]
    })

# Agrupar Planilhas por nome
for i, nome in enumerate(pl_table['nome']):
    pl_por_nome[nome].append({
        'origem': pl_table['origem'][i],
        'corretor': pl_table['corretor'][i],
        'situacao': pl_table['situação'][i]
    })

# Agrupar Prestações por nome
for i, nome in enumerate(pr_table['nome']):
    pr_por_nome[nome].append({
        'origem': pr_table['origem'][i],
        'nome_encontrado': pr_table['nome_encontrado'][i],
        'n_processo': pr_table['n_processo'][i]
    })

print("✓ Agrupamento por nome concluído")
print(f"  - Clientes com NF: {len(nf_por_nome)}")
print(f"  - Clientes com Planilha: {len(pl_por_nome)}")
print(f"  - Clientes com Prestação: {len(pr_por_nome)}")

In [ ]:
# 3 Construct final table with FULL JOIN

for nome in sorted(todos_nomes):
    # Adicionar nome
    final_table['nome'].append(nome)
    
    # ===== ORIGENS NF =====
    if nome in nf_por_nome:
        # Múltiplas NFs: adiciona lista de caminhos
        origens_nf = [nf['origem'] for nf in nf_por_nome[nome]]
        final_table['origem_nf'].append(origens_nf)
        
        # Data: pega da primeira NF (ou você pode concatenar todas)
        final_table['data_nf'].append(nf_por_nome[nome][0]['data'])
    else:
        final_table['origem_nf'].append(['Arquivo NF não encontrado'])
        final_table['data_nf'].append(None)
    
    # ===== ORIGENS PLANILHA =====
    if nome in pl_por_nome:
        # Múltiplas planilhas: adiciona lista de caminhos
        origens_pl = [pl['origem'] for pl in pl_por_nome[nome]]
        final_table['origem_pl'].append(origens_pl)
        
        # Situação e Corretor: pega da primeira (ou você pode verificar se são iguais)
        final_table['situação'].append(pl_por_nome[nome][0]['situacao'])
        final_table['corretor'].append(pl_por_nome[nome][0]['corretor'])
    else:
        final_table['origem_pl'].append(['Arquivo Planilha não encontrado'])
        final_table['situação'].append('Situação não encontrada')
        final_table['corretor'].append('Corretor não encontrado')
    
    # ===== ORIGENS PRESTAÇÃO =====
    if nome in pr_por_nome:
        # Múltiplas prestações: adiciona lista de caminhos
        origens_pr = [pr['origem'] for pr in pr_por_nome[nome]]
        final_table['origem_pr'].append(origens_pr)
        
        # Processo e nome encontrado: pega do primeiro
        final_table['n_processo'].append(pr_por_nome[nome][0]['n_processo'])
        final_table['nome_encontrado'].append(pr_por_nome[nome][0]['nome_encontrado'])
    else:
        final_table['origem_pr'].append(['Arquivo Prestação não encontrado'])
        final_table['n_processo'].append('Processo não encontrado')
        final_table['nome_encontrado'].append('Verificar prestação')

print(f"\n✓ Tabela final construída com {len(final_table['nome'])} clientes")

In [ ]:
# PASSO 4: Visualizar resumo e verificar inconsistências

# Converter para DataFrame para facilitar análise
df_final = pd.DataFrame(final_table)

print("="*60)
print("RESUMO DA TABELA FINAL")
print("="*60)
print(f"\nTotal de clientes: {len(df_final)}")
print(f"\nDimensões: {df_final.shape}")
print(f"\nColunas: {list(df_final.columns)}")

# Estatísticas de arquivos faltantes
print("\n" + "="*60)
print("ANÁLISE DE ARQUIVOS FALTANTES")
print("="*60)

nf_faltantes = df_final[df_final['origem_nf'].apply(lambda x: 'não encontrado' in str(x).lower())]
pl_faltantes = df_final[df_final['origem_pl'].apply(lambda x: 'não encontrado' in str(x).lower())]
pr_faltantes = df_final[df_final['origem_pr'].apply(lambda x: 'não encontrado' in str(x).lower())]

print(f"\n📄 Clientes SEM NF: {len(nf_faltantes)}")
print(f"📄 Clientes SEM Planilha: {len(pl_faltantes)}")
print(f"📄 Clientes SEM Prestação: {len(pr_faltantes)}")

# Clientes com múltiplos arquivos
multiplos_nf = df_final[df_final['origem_nf'].apply(lambda x: isinstance(x, list) and len(x) > 1 and 'não encontrado' not in str(x).lower())]
multiplos_pl = df_final[df_final['origem_pl'].apply(lambda x: isinstance(x, list) and len(x) > 1 and 'não encontrado' not in str(x).lower())]
multiplos_pr = df_final[df_final['origem_pr'].apply(lambda x: isinstance(x, list) and len(x) > 1 and 'não encontrado' not in str(x).lower())]

print("\n" + "="*60)
print("CLIENTES COM MÚLTIPLOS ARQUIVOS")
print("="*60)
print(f"\n📑 Clientes com >1 NF: {len(multiplos_nf)}")
if len(multiplos_nf) > 0:
    print("   Exemplos:")
    for idx in multiplos_nf.head(3).index:
        print(f"   - {df_final.loc[idx, 'nome']}: {len(df_final.loc[idx, 'origem_nf'])} arquivos")

print(f"\n📑 Clientes com >1 Planilha: {len(multiplos_pl)}")
if len(multiplos_pl) > 0:
    print("   Exemplos:")
    for idx in multiplos_pl.head(3).index:
        print(f"   - {df_final.loc[idx, 'nome']}: {len(df_final.loc[idx, 'origem_pl'])} arquivos")

print(f"\n📑 Clientes com >1 Prestação: {len(multiplos_pr)}")
if len(multiplos_pr) > 0:
    print("   Exemplos:")
    for idx in multiplos_pr.head(3).index:
        print(f"   - {df_final.loc[idx, 'nome']}: {len(df_final.loc[idx, 'origem_pr'])} arquivos")

# Mostrar primeiras linhas
print("\n" + "="*60)
print("PRIMEIRAS 10 LINHAS DA TABELA")
print("="*60)
print(df_final.head(10))

In [ ]:
df_final.head(30)

In [ ]:
# PASSO 5: Exportar tabela final para CSV/Excel

OUTPUT_FOLDER = str(PASTA_SAIDA / "RELATORIOS")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Exportar CSV
csv_path = os.path.join(OUTPUT_FOLDER, "tabela_final_clientes.csv")
df_final.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✓ CSV exportado: {csv_path}")

# Exportar Excel (com formatação melhor para listas)
excel_path = os.path.join(OUTPUT_FOLDER, "tabela_final_clientes.xlsx")

# Converter listas para strings para visualização no Excel
df_export = df_final.copy()
for col in ['origem_nf', 'origem_pl', 'origem_pr']:
    df_export[col] = df_export[col].apply(lambda x: '\n'.join(x) if isinstance(x, list) else str(x))

df_export.to_excel(excel_path, index=False, engine='openpyxl')
print(f"✓ Excel exportado: {excel_path}")

# Gerar relatórios de clientes faltantes
relatorio_faltantes = {
    'nome': [],
    'falta_nf': [],
    'falta_pl': [],
    'falta_pr': [],
    'total_faltantes': []
}

for idx, row in df_final.iterrows():
    falta_nf = 'não encontrado' in str(row['origem_nf']).lower()
    falta_pl = 'não encontrado' in str(row['origem_pl']).lower()
    falta_pr = 'não encontrado' in str(row['origem_pr']).lower()
    
    total_faltantes = sum([falta_nf, falta_pl, falta_pr])
    
    if total_faltantes > 0:  # Só adiciona se faltar algo
        relatorio_faltantes['nome'].append(row['nome'])
        relatorio_faltantes['falta_nf'].append('SIM' if falta_nf else 'NÃO')
        relatorio_faltantes['falta_pl'].append('SIM' if falta_pl else 'NÃO')
        relatorio_faltantes['falta_pr'].append('SIM' if falta_pr else 'NÃO')
        relatorio_faltantes['total_faltantes'].append(total_faltantes)

df_faltantes = pd.DataFrame(relatorio_faltantes)
faltantes_path = os.path.join(OUTPUT_FOLDER, "clientes_arquivos_faltantes.xlsx")
df_faltantes.to_excel(faltantes_path, index=False, engine='openpyxl')
print(f"✓ Relatório de faltantes exportado: {faltantes_path}")

print(f"\n{'='*60}")
print(f"EXPORTAÇÃO CONCLUÍDA!")
print(f"{'='*60}")

In [ ]:
'''# NF Relatory - Match corretor and situação from pl_table to nf_table else return none

for nf in nf_table['nome']:
    encontrado = False
    for i, pl in enumerate(pl_table['nome']):
        if nf == pl:
            nf_table['corretor'].append(pl_table['corretor'][i])
            nf_table['situação'].append(pl_table['situação'][i])
            encontrado = True            
            break
    if not encontrado:
        nf_table['corretor'].append(None)

for k, v in nf_table.items():
    print(f"{k}: {v}")'''

# PDF Merge

In [ ]:
# 1. Analisar e limpar páginas em branco de PDFs em uma pasta

def analisar_e_limpar_pdfs(caminho_da_pasta):
    """
    Analisa todos os arquivos PDF em uma pasta, relata suas propriedades
    e oferece a opção de remover páginas em branco.
    Filtra o relatório para exibir apenas arquivos com mais de uma página.

    Args:
        caminho_da_pasta (str): O caminho para a pasta contendo os arquivos PDF.
    """
    if not os.path.isdir(caminho_da_pasta):
        print(f"Erro: O caminho '{caminho_da_pasta}' não é uma pasta válida.")
        return

    # --- ESTRUTURAS DE DADOS PARA ARMAZENAR OS RESULTADOS ---
    relatorio_paginas = {}
    total_paginas_por_arquivo = {}
    arquivos_com_mais_de_uma_pagina = 0
    arquivos_com_paginas_em_branco = {}

    print(f"--- Iniciando Análise da Pasta: {caminho_da_pasta} ---\n")

    lista_de_arquivos = [f for f in os.listdir(caminho_da_pasta) if f.lower().endswith('.pdf')]

    if not lista_de_arquivos:
        print("Nenhum arquivo PDF encontrado na pasta.")
        return

    # --- FASE 1: ANÁLISE DOS ARQUIVOS ---
    for nome_arquivo in lista_de_arquivos:
        caminho_completo = os.path.join(caminho_da_pasta, nome_arquivo)
        
        try:
            doc = fitz.open(caminho_completo)
            total_paginas = doc.page_count
            total_paginas_por_arquivo[nome_arquivo] = total_paginas
            paginas_em_branco_indices = []
            
            if total_paginas > 1:
                arquivos_com_mais_de_uma_pagina += 1

            for i in range(total_paginas):
                page = doc.load_page(i)
                if not page.get_text() and not page.get_images():
                    paginas_em_branco_indices.append(i)

            paginas_validas = total_paginas - len(paginas_em_branco_indices)
            relatorio_paginas[nome_arquivo] = paginas_validas
            
            if paginas_em_branco_indices:
                arquivos_com_paginas_em_branco[nome_arquivo] = [p + 1 for p in paginas_em_branco_indices]

            doc.close()

        except Exception as e:
            print(f"Não foi possível processar o arquivo '{nome_arquivo}'. Erro: {e}")

    # --- FASE 2: APRESENTAÇÃO DOS RESULTADOS ---
    print("--- Relatório da Análise Concluído ---\n")
    
    print("1. Quantidade de páginas válidas (não-brancas) por arquivo (somente arquivos com >1 página):")
    if relatorio_paginas:
        arquivos_filtrados_exibidos = False
        for nome, qtd in relatorio_paginas.items():
            if total_paginas_por_arquivo.get(nome, 0) > 1:
                print(f"  - {nome}: {qtd} página(s)")
                arquivos_filtrados_exibidos = True
        if not arquivos_filtrados_exibidos:
             print("  Nenhum arquivo com mais de uma página foi encontrado para listar.")
    else:
        print("  Nenhum arquivo processado.")
    print("-" * 30)

    print(f"2. Total de arquivos com mais de uma página: {arquivos_com_mais_de_uma_pagina}")
    print("-" * 30)
    
    print("3. Arquivos que contêm páginas em branco:")
    if arquivos_com_paginas_em_branco:
        for nome, paginas in arquivos_com_paginas_em_branco.items():
            print(f"  - {nome} (Páginas em branco: {paginas})")
    else:
        print("  Nenhum arquivo com páginas em branco foi encontrado.")
    print("-" * 30)

    print("\n--- Verificação Manual Completa ---")
    
    # --- FASE 3: AÇÃO DE APAGAR PÁGINAS (COM CONFIRMAÇÃO) ---
    if not arquivos_com_paginas_em_branco:
        print("\nProcesso finalizado. Nenhuma ação de modificação necessária.")
        return

    prosseguir = input("\nDeseja apagar as páginas em branco listadas acima? (s/n): ").lower()

    if prosseguir == 's':
        print("\n--- Iniciando a remoção de páginas em branco ---")
        
        pasta_saida = os.path.join(caminho_da_pasta, "arquivos_limpos")
        os.makedirs(pasta_saida, exist_ok=True)
        print(f"Os arquivos modificados serão salvos em: '{pasta_saida}'")

        for nome_arquivo, paginas in arquivos_com_paginas_em_branco.items():
            try:
                caminho_original = os.path.join(caminho_da_pasta, nome_arquivo)
                doc = fitz.open(caminho_original)
                
                indices_para_remover = sorted([p - 1 for p in paginas], reverse=True)
                
                for indice in indices_para_remover:
                    doc.delete_page(indice)
                
                caminho_novo_arquivo = os.path.join(pasta_saida, nome_arquivo)
                doc.save(caminho_novo_arquivo, garbage=4, deflate=True)
                doc.close()
                print(f"  - Páginas removidas de '{nome_arquivo}'. Nova versão salva.")

            except Exception as e:
                print(f"  - Erro ao modificar o arquivo '{nome_arquivo}': {e}")
        
        print("\n--- Processo de limpeza concluído! ---")
    else:
        print("\nNenhuma modificação foi realizada. Processo encerrado.")


# Exemplo de uso:
# analisar_e_limpar_pdfs(PATH_PLANILHAS_PRESTACOES)
# analisar_e_limpar_pdfs(PATH_NF)



In [ ]:
# 2. Remover última página de um arquivo PDF específico

def apagar_ultima_pagina(diretorio, nome_arquivo):
    """
    Remove a última página de um arquivo PDF específico.
    Este método usa um arquivo temporário para garantir a compatibilidade e segurança.
    
    Args:
        diretorio (str): Caminho do diretório onde está o arquivo
        nome_arquivo (str): Nome do arquivo PDF
    """
    caminho_completo = os.path.join(diretorio, nome_arquivo)
    nome_temp = os.path.join(diretorio, "temp_" + nome_arquivo)

    if not os.path.isfile(caminho_completo):
        print(f"\nERRO: O arquivo '{nome_arquivo}' não foi encontrado no diretório especificado.")
        return

    try:
        doc = fitz.open(caminho_completo)
        total_paginas = doc.page_count
        
        if total_paginas <= 1:
            print(f"\nAVISO: O arquivo '{nome_arquivo}' possui apenas uma página. Nenhuma ação foi tomada.")
            doc.close()
            return

        print(f"\nProcessando '{nome_arquivo}'...")
        print(f"O arquivo possui {total_paginas} páginas. A última página (nº {total_paginas}) será removida.")

        doc.delete_page(total_paginas - 1)
        doc.save(nome_temp, garbage=4, deflate=True)
        doc.close()

        os.remove(caminho_completo)
        os.rename(nome_temp, caminho_completo)

        print(f"\nSUCESSO! A última página do arquivo '{nome_arquivo}' foi removida.")

    except Exception as e:
        print(f"\nERRO: Ocorreu um problema ao processar o arquivo '{nome_arquivo}': {e}")
        if os.path.exists(nome_temp):
            os.remove(nome_temp)


# Exemplo de uso:
# apagar_ultima_pagina(PATH_PLANILHAS_PRESTACOES, "nome_do_arquivo.pdf")
# apagar_ultima_pagina(PATH_NF, "nome_do_arquivo.pdf")

In [ ]:
# 3 Merge cleaned PDFs per client

# Definir pasta de destino
PASTA_DESTINO = str(PASTA_SAIDA / "ARQUIVOS_TRATADOS")
os.makedirs(PASTA_DESTINO, exist_ok=True)

print("="*60)
print("INICIANDO MESCLAGEM DE PDFs POR CLIENTE")
print("="*60)

contador_sucesso = 0
contador_erro = 0
clientes_com_erro = []

for idx, row in df_final.iterrows():
    nome_cliente = row['nome']
    
    try:
        merger = PyPDF2.PdfMerger()
        arquivos_adicionados = 0
        
        # ORDEM: 1. Prestação, 2. Planilha, 3. NF
        ordem_merge = [
            ('Prestação', row['origem_pr']),
            ('Planilha', row['origem_pl']),
            ('NF', row['origem_nf'])
        ]
        
        for tipo, origens in ordem_merge:
            # Verificar se não é arquivo faltante
            if isinstance(origens, list) and 'não encontrado' not in str(origens).lower():
                # Adicionar todos os arquivos deste tipo (suporta múltiplos)
                for origem in origens:
                    if os.path.exists(origem):
                        merger.append(origem)
                        arquivos_adicionados += 1
                        print(f"  ✓ {tipo}: {os.path.basename(origem)}")
                    else:
                        print(f"  ⚠ {tipo}: arquivo não existe - {origem}")
        
        # Só salvar se adicionou pelo menos um arquivo
        if arquivos_adicionados > 0:
            caminho_saida = os.path.join(PASTA_DESTINO, f"{nome_cliente}.pdf")
            merger.write(caminho_saida)
            merger.close()
            print(f"✓ PDF mesclado criado: {nome_cliente}.pdf ({arquivos_adicionados} arquivos)\n")
            contador_sucesso += 1
        else:
            print(f"⚠ Nenhum arquivo encontrado para {nome_cliente} - PDF não criado\n")
            clientes_com_erro.append(nome_cliente)
            contador_erro += 1
            merger.close()
            
    except Exception as e:
        print(f"❌ ERRO ao processar {nome_cliente}: {e}\n")
        clientes_com_erro.append(nome_cliente)
        contador_erro += 1

# Final Relatory

In [ ]:
print("="*60)
print("RESUMO DA MESCLAGEM")
print("="*60)
print(f"✓ PDFs criados com sucesso: {contador_sucesso}")
print(f"❌ Clientes com erro/sem arquivos: {contador_erro}")
print(f"\nPasta de destino: {PASTA_DESTINO}")

if clientes_com_erro:
    print(f"\n⚠ Clientes que não geraram PDF:")
    for cliente in clientes_com_erro[:10]:  # Mostrar primeiros 10
        print(f"  - {cliente}")
    if len(clientes_com_erro) > 10:
        print(f"  ... e mais {len(clientes_com_erro) - 10} clientes")

print("="*60)

In [ ]:
# Verificação rápida das categorias após normalização
cols = [c for c in ['situação','status'] if 'df_final' in globals() and c in df_final.columns]
if cols:
    for c in cols:
        print(f"Valores únicos em '{c}':")
        print(sorted(set(df_final[c].astype(str))))
else:
    print("df_final não possui colunas 'situação' ou 'status'.")